In [1]:
import os
import glob
import numpy as np
from astropy.io import fits
from scipy import ndimage
from scipy.ndimage import gaussian_filter
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

input_folder = r"test"
output_folder = r"featurestest"

os.makedirs(output_folder, exist_ok=True)

def validate_data(data, fname):
    """Validate that data is 2D and has proper dimensions"""
    if data is None:
        raise ValueError("Data is None")
    
    if data.ndim != 2:
        raise ValueError(f"Data has {data.ndim} dimensions, expected 2D")
    
    if data.shape[0] < 10 or data.shape[1] < 10:
        raise ValueError(f"Data too small: {data.shape}")
    
    if np.all(data == 0):
        raise ValueError("Data contains only zeros")
    
    return True

def create_rings(data, n_rings=10):
    """Create concentric square rings from center"""
    cy, cx = data.shape[0] // 2, data.shape[1] // 2
    y, x = np.ogrid[:data.shape[0], :data.shape[1]]
    dist = np.maximum(np.abs(x - cx), np.abs(y - cy))
    
    max_dist = min(cx, cy, data.shape[0] - cy, data.shape[1] - cx)
    ring_width = max_dist / n_rings
    
    rings = []
    for i in range(n_rings):
        inner = i * ring_width
        outer = (i + 1) * ring_width
        mask = (dist >= inner) & (dist < outer)
        rings.append(mask)
    
    return rings

def calc_descent_average(data, rings):
    """DA - measures light distribution across rings"""
    ring_intensities = []
    for ring in rings:
        if np.any(ring):
            ring_intensities.append(np.mean(data[ring]))
        else:
            ring_intensities.append(0)
    
    # normalize and skip first 2 rings (bulge)
    intensities = np.array(ring_intensities[2:])
    if len(intensities) == 0 or intensities.max() == 0:
        return 0
    
    intensities = intensities / intensities.max()
    
    # area under curve
    da = np.trapz(intensities) / len(intensities)
    return da

def calc_descent_variance(data, rings):
    """DV - measures variance within rings"""
    ring_variances = []
    for ring in rings:
        if np.any(ring) and np.sum(ring) > 1:
            ring_variances.append(np.var(data[ring]))
        else:
            ring_variances.append(0)
    
    # normalize
    variances = np.array(ring_variances)
    if variances.max() > 0:
        variances = variances / variances.max()
    
    dv = np.mean(variances)
    return dv

def calc_asymmetry(data):
    """A - simplified asymmetry measure"""
    try:
        rotated = ndimage.rotate(data, 180, reshape=False, order=1)
        diff = np.abs(data - rotated)
        
        total_intensity = np.sum(np.abs(data))
        if total_intensity > 0:
            asymmetry = np.sum(diff) / total_intensity
        else:
            asymmetry = 0
        
        return asymmetry
    except Exception as e:
        return 0

def calc_clumpiness(data):
    """S - detects non-smooth structures (spiral arms)"""
    try:
        # smooth with gaussian
        smoothed = ndimage.gaussian_filter(data, sigma=7)
        
        # difference image
        diff = data - smoothed
        
        # remove negatives and center (top 1%)
        diff[diff < 0] = 0
        threshold = np.percentile(data, 99)
        diff[data >= threshold] = 0
        
        total_intensity = np.sum(np.abs(data))
        if total_intensity > 0:
            clumpiness = np.sum(diff) / total_intensity
        else:
            clumpiness = 0
        
        return clumpiness
    except Exception as e:
        return 0

def calc_concentration(data):
    """C - light concentration (inverse concentration index R50/R90)"""
    try:
        cy, cx = data.shape[0] // 2, data.shape[1] // 2
        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        r = np.sqrt((x - cx)**2 + (y - cy)**2)
        
        total = np.sum(data)
        if total == 0:
            return 0, 0, 0
        
        # find radii containing 50% and 90% of light
        sorted_idx = np.argsort(r.flatten())
        cumsum = np.cumsum(data.flatten()[sorted_idx])
        cumsum = cumsum / total
        
        r50_idx = np.searchsorted(cumsum, 0.5)
        r90_idx = np.searchsorted(cumsum, 0.9)
        
        r50 = r.flatten()[sorted_idx[r50_idx]] if r50_idx < len(sorted_idx) else 0
        r90 = r.flatten()[sorted_idx[r90_idx]] if r90_idx < len(sorted_idx) else 0
        
        # Concentration C = r50/r90 (inverse concentration index)
        if r90 > 0:
            concentration = r50 / r90
        else:
            concentration = 0
        
        return concentration, r50, r90
    except Exception as e:
        return 0, 0, 0

def calc_bar_to_bulge_ratio(data):
    """
    Bar-to-bulge ratio - measures elongation in central region
    Higher values indicate presence of bar structure
    """
    try:
        cy, cx = data.shape[0] // 2, data.shape[1] // 2
        
        # Focus on central 40% of image (where bar would be)
        h, w = data.shape
        y_start, y_end = int(cy - h*0.2), int(cy + h*0.2)
        x_start, x_end = int(cx - w*0.2), int(cx + w*0.2)
        
        # Ensure valid bounds
        y_start = max(0, y_start)
        y_end = min(h, y_end)
        x_start = max(0, x_start)
        x_end = min(w, x_end)
        
        central = data[y_start:y_end, x_start:x_end]
        
        if central.size == 0 or central.shape[0] < 5 or central.shape[1] < 5:
            return 0
        
        # Calculate second moments to measure elongation
        y, x = np.mgrid[:central.shape[0], :central.shape[1]]
        total = np.sum(central)
        
        if total == 0:
            return 0
        
        # Center of mass
        x_cm = np.sum(x * central) / total
        y_cm = np.sum(y * central) / total
        
        # Second moments
        mxx = np.sum(central * (x - x_cm)**2) / total
        myy = np.sum(central * (y - y_cm)**2) / total
        mxy = np.sum(central * (x - x_cm) * (y - y_cm)) / total
        
        # Calculate ellipticity (bar indicator)
        trace = mxx + myy
        det = mxx * myy - mxy**2
        
        if det > 0 and trace > 0:
            # Ratio of semi-major to semi-minor axis
            discriminant = trace**2 - 4*det
            if discriminant >= 0:
                lambda_plus = 0.5 * (trace + np.sqrt(discriminant))
                lambda_minus = 0.5 * (trace - np.sqrt(discriminant))
                
                if lambda_minus > 0:
                    bar_ratio = np.sqrt(lambda_plus / lambda_minus)
                else:
                    bar_ratio = 0
            else:
                bar_ratio = 0
        else:
            bar_ratio = 0
        
        return bar_ratio
    except Exception as e:
        return 0

def calc_central_concentration(data):
    """
    Central concentration in inner 10% vs outer regions
    Helps distinguish nuclear sphere in unbarred spirals
    """
    try:
        cy, cx = data.shape[0] // 2, data.shape[1] // 2
        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        r = np.sqrt((x - cx)**2 + (y - cy)**2)
        
        max_r = min(cx, cy)
        inner_r = max_r * 0.1
        
        inner_mask = r <= inner_r
        outer_mask = r > inner_r
        
        inner_sum = np.sum(data[inner_mask])
        outer_sum = np.sum(data[outer_mask])
        
        if outer_sum > 0:
            central_conc = inner_sum / outer_sum
        else:
            central_conc = 0
        
        return central_conc
    except Exception as e:
        return 0

def calc_spiral_arm_strength(data):
    """
    Measures strength of spiral arm pattern
    Uses multiple scale smoothing to detect arm structures
    """
    try:
        # Smooth at different scales
        smooth_small = gaussian_filter(data, sigma=3)
        smooth_large = gaussian_filter(data, sigma=10)
        
        # Difference highlights medium-scale structures (spiral arms)
        arm_structure = smooth_small - smooth_large
        arm_structure[arm_structure < 0] = 0
        
        total = np.sum(data)
        if total > 0:
            arm_strength = np.sum(arm_structure) / total
        else:
            arm_strength = 0
        
        return arm_strength
    except Exception as e:
        return 0

def calc_gini_coefficient(data):
    """
    Gini coefficient - measures light distribution inequality
    Higher values indicate more concentrated light distribution
    """
    try:
        flat_data = data.flatten()
        flat_data = flat_data[flat_data > 0]  # Remove zeros
        
        if len(flat_data) == 0:
            return 0
        
        sorted_data = np.sort(flat_data)
        n = len(sorted_data)
        index = np.arange(1, n + 1)
        
        gini = (2 * np.sum(index * sorted_data)) / (n * np.sum(sorted_data)) - (n + 1) / n
        
        return gini
    except Exception as e:
        return 0

def calc_ellipticity(data):
    """
    Overall ellipticity of the galaxy
    Barred spirals may show higher ellipticity
    """
    try:
        cy, cx = data.shape[0] // 2, data.shape[1] // 2
        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        
        total = np.sum(data)
        if total == 0:
            return 0
        
        # Second moments
        mxx = np.sum(data * (x - cx)**2) / total
        myy = np.sum(data * (y - cy)**2) / total
        mxy = np.sum(data * (x - cx) * (y - cy)) / total
        
        # Ellipticity
        trace = mxx + myy
        det = mxx * myy - mxy**2
        
        if det > 0 and trace > 0:
            ellipticity = 1 - np.sqrt(4 * det / trace**2)
        else:
            ellipticity = 0
        
        return ellipticity
    except Exception as e:
        return 0

def calc_petrosian_radius(data):
    """
    Petrosian radius - radius where surface brightness profile drops
    Different for barred vs unbarred spirals
    """
    try:
        cy, cx = data.shape[0] // 2, data.shape[1] // 2
        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        r = np.sqrt((x - cx)**2 + (y - cy)**2)
        
        max_r = min(cx, cy)
        if max_r < 10:
            return 0
        
        radii = np.linspace(0, max_r, 50)
        
        eta_values = []
        for radius in radii[1:]:
            annulus_mask = (r >= radius - max_r/50) & (r < radius)
            circle_mask = r < radius
            
            if np.any(annulus_mask) and np.any(circle_mask):
                annulus_brightness = np.mean(data[annulus_mask])
                circle_brightness = np.mean(data[circle_mask])
                
                if circle_brightness > 0:
                    eta = annulus_brightness / circle_brightness
                    eta_values.append(eta)
                else:
                    eta_values.append(0)
            else:
                eta_values.append(0)
        
        # Petrosian radius where eta = 0.2
        eta_values = np.array(eta_values)
        if len(eta_values) > 0 and np.any(eta_values < 0.2):
            petro_idx = np.where(eta_values < 0.2)[0][0]
            petrosian_r = radii[petro_idx + 1]
        else:
            petrosian_r = max_r
        
        return petrosian_r / max_r  # Normalized
    except Exception as e:
        return 0

# Process all preprocessed files
fits_files = glob.glob(os.path.join(input_folder, "*.fits"))
total = len(fits_files)
print(f"Extracting enhanced features from {total} files")
print("="*60)

features_list = []
error_count = 0

for i, fpath in enumerate(fits_files, 1):
    fname = os.path.basename(fpath)
    
    if i % max(1, total // 10) == 0:
        print(f"Progress: {i}/{total} ({100*i//total}%) - Errors: {error_count}")
    
    try:
        with fits.open(fpath) as hdul:
            # Get primary data
            if len(hdul) > 0:
                data = hdul[0].data
            else:
                raise ValueError("No data in FITS file")
            
            # Validate data
            validate_data(data, fname)
            
            # Normalize data for consistent measurements
            if data.max() > 0:
                data = data.astype(float) / data.max()
            else:
                raise ValueError("Max value is zero")
            
            # Create rings
            rings = create_rings(data, n_rings=10)
            
            # Calculate all features
            da = calc_descent_average(data, rings)
            dv = calc_descent_variance(data, rings)
            asym = calc_asymmetry(data)
            clump = calc_clumpiness(data)
            conc, r50, r90 = calc_concentration(data)
            
            # New features for bar classification
            bar_bulge = calc_bar_to_bulge_ratio(data)
            central_conc = calc_central_concentration(data)
            spiral_strength = calc_spiral_arm_strength(data)
            gini = calc_gini_coefficient(data)
            ellip = calc_ellipticity(data)
            petro = calc_petrosian_radius(data)
            
            features_list.append({
                'filename': fname,
                # Original features
                'DA': da,
                'DV': dv,
                'Asymmetry': asym,
                'Clumpiness': clump,
                'Concentration': conc,
                'R50': r50,
                'R90': r90,
                # Bar-specific features
                'Bar_Bulge_Ratio': bar_bulge,
                'Central_Concentration': central_conc,
                'Spiral_Arm_Strength': spiral_strength,
                # Additional morphological features
                'Gini': gini,
                'Ellipticity': ellip,
                'Petrosian_Radius': petro
            })
            
    except Exception as e:
        error_count += 1
        if error_count <= 10:  # Print first 10 errors only
            print(f"  ERROR: {fname} - {str(e)[:80]}")
        elif error_count == 11:
            print(f"  ... suppressing further error messages ...")

# Save features to CSV
df = pd.DataFrame(features_list)
csv_path = os.path.join(output_folder, "galaxy_features_enhanced.csv")
df.to_csv(csv_path, index=False)

print("\n" + "="*60)
print("Enhanced feature extraction complete!")
print(f"Successfully extracted: {len(df)} feature vectors")
print(f"Failed: {error_count} files")
print(f"Success rate: {100*len(df)/total:.1f}%")
print(f"\nFeatures extracted:")
print("  - Original: DA, DV, Asymmetry, Clumpiness, Concentration, R50, R90")
print("  - Bar-specific: Bar_Bulge_Ratio, Central_Concentration, Spiral_Arm_Strength")
print("  - Additional: Gini, M20, Ellipticity, Petrosian_Radius")
print(f"\nTotal features: {len(df.columns) - 1}")
print(f"\nSaved to: {csv_path}")
print(f"\nSample statistics:")
print(df.describe())

Extracting enhanced features from 117 files
Progress: 11/117 (9%) - Errors: 0
Progress: 22/117 (18%) - Errors: 0
Progress: 33/117 (28%) - Errors: 0
Progress: 44/117 (37%) - Errors: 0
Progress: 55/117 (47%) - Errors: 0
Progress: 66/117 (56%) - Errors: 0
Progress: 77/117 (65%) - Errors: 0
Progress: 88/117 (75%) - Errors: 0
Progress: 99/117 (84%) - Errors: 0
Progress: 110/117 (94%) - Errors: 0

Enhanced feature extraction complete!
Successfully extracted: 117 feature vectors
Failed: 0 files
Success rate: 100.0%

Features extracted:
  - Original: DA, DV, Asymmetry, Clumpiness, Concentration, R50, R90
  - Bar-specific: Bar_Bulge_Ratio, Central_Concentration, Spiral_Arm_Strength
  - Additional: Gini, M20, Ellipticity, Petrosian_Radius

Total features: 13

Saved to: featurestest\galaxy_features_enhanced.csv

Sample statistics:
               DA          DV   Asymmetry  Clumpiness  Concentration  \
count  117.000000  117.000000  117.000000  117.000000     117.000000   
mean     0.353047    0.3

In [3]:
import os
import glob
import numpy as np
from astropy.io import fits
from scipy import ndimage
from scipy.ndimage import gaussian_filter
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

input_folder = r"train"
output_folder = r"featurestrain"

os.makedirs(output_folder, exist_ok=True)

def validate_data(data, fname):
    """Validate that data is 2D and has proper dimensions"""
    if data is None:
        raise ValueError("Data is None")
    
    if data.ndim != 2:
        raise ValueError(f"Data has {data.ndim} dimensions, expected 2D")
    
    if data.shape[0] < 10 or data.shape[1] < 10:
        raise ValueError(f"Data too small: {data.shape}")
    
    if np.all(data == 0):
        raise ValueError("Data contains only zeros")
    
    return True
def calc_color_proxy(data):
    """
    Color Proxy - estimates galaxy color (blue vs red tendency) from brightness distribution.
    Since FITS is single-band, we use texture and gradient smoothness as a proxy.
    Bluer galaxies (younger, star-forming) tend to have more clumpy, high-frequency light.
    Redder galaxies (older) are smoother.
    """
    try:
        # Compute Laplacian variance (texture measure)
        laplace = ndimage.laplace(data)
        texture = np.var(laplace)
        
        # Normalize
        norm_texture = texture / (np.mean(data) + 1e-6)
        # Higher texture → bluer proxy
        color_proxy = np.tanh(norm_texture * 0.5)
        return color_proxy
    except:
        return 0


def calc_gas_concentration(data):
    """
    Gas Concentration Proxy - mimics how gas is funneled toward the center in barred galaxies.
    We measure how much flux (light) lies in inner 30% of radius vs total light.
    """
    try:
        cy, cx = data.shape[0] // 2, data.shape[1] // 2
        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        r = np.sqrt((x - cx)**2 + (y - cy)**2)
        r_norm = r / r.max()
        
        inner = r_norm <= 0.3
        outer = r_norm > 0.3
        
        inner_sum = np.sum(data[inner])
        total_sum = np.sum(data)
        if total_sum == 0:
            return 0
        
        gas_conc = inner_sum / total_sum
        return gas_conc
    except:
        return 0


def calc_radial_profile_slope(data):
    """
    Radial Profile Slope - slope of mean intensity with radius (log scale).
    Barred galaxies tend to have flatter (shallower) slopes due to redistributed material.
    """
    try:
        cy, cx = data.shape[0] // 2, data.shape[1] // 2
        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        r = np.sqrt((x - cx)**2 + (y - cy)**2)
        r_flat = r.flatten()
        d_flat = data.flatten()
        
        # Bin radii
        bins = np.linspace(0, r_flat.max(), 30)
        means = ndimage.mean(d_flat, labels=np.digitize(r_flat, bins), index=np.arange(1, len(bins)+1))
        valid = (means > 0)
        
        if np.sum(valid) < 5:
            return 0
        
        r_vals = bins[valid]
        I_vals = means[valid]
        
        # Linear fit in log-log space
        slope, _ = np.polyfit(np.log10(r_vals+1e-6), np.log10(I_vals+1e-6), 1)
        return slope
    except:
        return 0


def calc_bar_strength_fourier(data):
    """
    Bar Fourier Strength - measures 2-fold (m=2) azimuthal mode strength in intensity.
    Barred galaxies show strong m=2 component.
    """
    try:
        cy, cx = data.shape[0] // 2, data.shape[1] // 2
        y, x = np.indices(data.shape)
        r = np.sqrt((x - cx)**2 + (y - cy)**2)
        theta = np.arctan2(y - cy, x - cx)
        
        r_max = np.max(r)
        r_bins = np.linspace(0, r_max, 20)
        A2_list = []
        
        for i in range(len(r_bins)-1):
            annulus = (r >= r_bins[i]) & (r < r_bins[i+1])
            if np.any(annulus):
                I = data[annulus]
                phi = theta[annulus]
                # Fourier m=2 component
                A2 = np.abs(np.sum(I * np.exp(2j * phi))) / np.sum(I)
                A2_list.append(A2)
        
        if len(A2_list) == 0:
            return 0
        
        bar_strength = np.mean(A2_list)
        return bar_strength
    except:
        return 0


def calc_spiral_arm_pitch_angle(data):
    """
    Spiral Arm Pitch Angle - rough proxy based on radial gradient orientation dispersion.
    Barred spirals tend to have tighter arms (smaller pitch).
    """
    try:
        # Compute gradients
        gy, gx = np.gradient(gaussian_filter(data, 2))
        angles = np.degrees(np.arctan2(gy, gx))
        valid = (data > np.percentile(data, 70))
        angle_dispersion = np.std(angles[valid])
        
        # Higher dispersion → looser arms → higher pitch angle
        pitch_proxy = np.tanh(angle_dispersion / 60)
        return pitch_proxy
    except:
        return 0


def calc_bulge_to_total(data):
    """
    Bulge-to-Total Light Ratio (B/T) - central light fraction proxy.
    Barred galaxies usually have slightly higher B/T due to bar buildup.
    """
    try:
        cy, cx = data.shape[0] // 2, data.shape[1] // 2
        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        r = np.sqrt((x - cx)**2 + (y - cy)**2)
        r_norm = r / r.max()
        
        bulge = np.sum(data[r_norm <= 0.2])
        total = np.sum(data)
        if total == 0:
            return 0
        
        bt = bulge / total
        return bt
    except:
        return 0


def calc_angular_momentum_proxy(data):
    """
    Angular Momentum Proxy - approximates rotational symmetry.
    Uses correlation between opposite quadrants; higher correlation → higher angular momentum transfer.
    """
    try:
        cy, cx = data.shape[0] // 2, data.shape[1] // 2
        q1 = data[:cy, :cx]
        q3 = data[cy:, cx:]
        
        if q1.size == 0 or q3.size == 0:
            return 0
        
        q1 = (q1 - np.mean(q1)) / (np.std(q1) + 1e-6)
        q3 = (q3 - np.mean(q3)) / (np.std(q3) + 1e-6)
        q3_resized = ndimage.zoom(q3, (q1.shape[0]/q3.shape[0], q1.shape[1]/q3.shape[1]))
        
        corr = np.mean(q1 * q3_resized)
        return corr
    except:
        return 0

def create_rings(data, n_rings=10):
    """Create concentric square rings from center"""
    cy, cx = data.shape[0] // 2, data.shape[1] // 2
    y, x = np.ogrid[:data.shape[0], :data.shape[1]]
    dist = np.maximum(np.abs(x - cx), np.abs(y - cy))
    
    max_dist = min(cx, cy, data.shape[0] - cy, data.shape[1] - cx)
    ring_width = max_dist / n_rings
    
    rings = []
    for i in range(n_rings):
        inner = i * ring_width
        outer = (i + 1) * ring_width
        mask = (dist >= inner) & (dist < outer)
        rings.append(mask)
    
    return rings

def calc_descent_average(data, rings):
    """DA - measures light distribution across rings"""
    ring_intensities = []
    for ring in rings:
        if np.any(ring):
            ring_intensities.append(np.mean(data[ring]))
        else:
            ring_intensities.append(0)
    
    # normalize and skip first 2 rings (bulge)
    intensities = np.array(ring_intensities[2:])
    if len(intensities) == 0 or intensities.max() == 0:
        return 0
    
    intensities = intensities / intensities.max()
    
    # area under curve
    da = np.trapz(intensities) / len(intensities)
    return da

def calc_descent_variance(data, rings):
    """DV - measures variance within rings"""
    ring_variances = []
    for ring in rings:
        if np.any(ring) and np.sum(ring) > 1:
            ring_variances.append(np.var(data[ring]))
        else:
            ring_variances.append(0)
    
    # normalize
    variances = np.array(ring_variances)
    if variances.max() > 0:
        variances = variances / variances.max()
    
    dv = np.mean(variances)
    return dv

def calc_asymmetry(data):
    """A - asymmetry measure at both 90 and 180 degrees"""
    try:
        # 180-degree asymmetry (original)
        rotated_180 = ndimage.rotate(data, 180, reshape=False, order=1)
        diff_180 = np.abs(data - rotated_180)
        
        total_intensity = np.sum(np.abs(data))
        if total_intensity > 0:
            asymmetry_180 = np.sum(diff_180) / total_intensity
        else:
            asymmetry_180 = 0
        
        # 90-degree asymmetry (new - for bar detection)
        rotated_90 = ndimage.rotate(data, 90, reshape=False, order=1)
        diff_90 = np.abs(data - rotated_90)
        
        if total_intensity > 0:
            asymmetry_90 = np.sum(diff_90) / total_intensity
        else:
            asymmetry_90 = 0
        
        return asymmetry_180, asymmetry_90
    except Exception as e:
        return 0, 0

def calc_clumpiness(data):
    """S - detects non-smooth structures (spiral arms)"""
    try:
        # smooth with gaussian
        smoothed = ndimage.gaussian_filter(data, sigma=7)
        
        # difference image
        diff = data - smoothed
        
        # remove negatives and center (top 1%)
        diff[diff < 0] = 0
        threshold = np.percentile(data, 99)
        diff[data >= threshold] = 0
        
        total_intensity = np.sum(np.abs(data))
        if total_intensity > 0:
            clumpiness = np.sum(diff) / total_intensity
        else:
            clumpiness = 0
        
        return clumpiness
    except Exception as e:
        return 0

def calc_concentration(data):
    """C - light concentration (inverse concentration index R50/R90)"""
    try:
        cy, cx = data.shape[0] // 2, data.shape[1] // 2
        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        r = np.sqrt((x - cx)**2 + (y - cy)**2)
        
        total = np.sum(data)
        if total == 0:
            return 0, 0, 0
        
        # find radii containing 50% and 90% of light
        sorted_idx = np.argsort(r.flatten())
        cumsum = np.cumsum(data.flatten()[sorted_idx])
        cumsum = cumsum / total
        
        r50_idx = np.searchsorted(cumsum, 0.5)
        r90_idx = np.searchsorted(cumsum, 0.9)
        
        r50 = r.flatten()[sorted_idx[r50_idx]] if r50_idx < len(sorted_idx) else 0
        r90 = r.flatten()[sorted_idx[r90_idx]] if r90_idx < len(sorted_idx) else 0
        
        # Concentration C = r50/r90 (inverse concentration index)
        if r90 > 0:
            concentration = r50 / r90
        else:
            concentration = 0
        
        return concentration, r50, r90
    except Exception as e:
        return 0, 0, 0

def calc_bar_to_bulge_ratio(data):
    """
    Bar-to-bulge ratio - measures elongation in central region
    Higher values indicate presence of bar structure
    """
    try:
        cy, cx = data.shape[0] // 2, data.shape[1] // 2
        
        # Focus on central 40% of image (where bar would be)
        h, w = data.shape
        y_start, y_end = int(cy - h*0.2), int(cy + h*0.2)
        x_start, x_end = int(cx - w*0.2), int(cx + w*0.2)
        
        # Ensure valid bounds
        y_start = max(0, y_start)
        y_end = min(h, y_end)
        x_start = max(0, x_start)
        x_end = min(w, x_end)
        
        central = data[y_start:y_end, x_start:x_end]
        
        if central.size == 0 or central.shape[0] < 5 or central.shape[1] < 5:
            return 0
        
        # Calculate second moments to measure elongation
        y, x = np.mgrid[:central.shape[0], :central.shape[1]]
        total = np.sum(central)
        
        if total == 0:
            return 0
        
        # Center of mass
        x_cm = np.sum(x * central) / total
        y_cm = np.sum(y * central) / total
        
        # Second moments
        mxx = np.sum(central * (x - x_cm)**2) / total
        myy = np.sum(central * (y - y_cm)**2) / total
        mxy = np.sum(central * (x - x_cm) * (y - y_cm)) / total
        
        # Calculate ellipticity (bar indicator)
        trace = mxx + myy
        det = mxx * myy - mxy**2
        
        if det > 0 and trace > 0:
            # Ratio of semi-major to semi-minor axis
            discriminant = trace**2 - 4*det
            if discriminant >= 0:
                lambda_plus = 0.5 * (trace + np.sqrt(discriminant))
                lambda_minus = 0.5 * (trace - np.sqrt(discriminant))
                
                if lambda_minus > 0:
                    bar_ratio = np.sqrt(lambda_plus / lambda_minus)
                else:
                    bar_ratio = 0
            else:
                bar_ratio = 0
        else:
            bar_ratio = 0
        
        return bar_ratio
    except Exception as e:
        return 0

def calc_central_concentration(data):
    """
    Central concentration in inner 10% vs outer regions
    Helps distinguish nuclear sphere in unbarred spirals
    """
    try:
        cy, cx = data.shape[0] // 2, data.shape[1] // 2
        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        r = np.sqrt((x - cx)**2 + (y - cy)**2)
        
        max_r = min(cx, cy)
        inner_r = max_r * 0.1
        
        inner_mask = r <= inner_r
        outer_mask = r > inner_r
        
        inner_sum = np.sum(data[inner_mask])
        outer_sum = np.sum(data[outer_mask])
        
        if outer_sum > 0:
            central_conc = inner_sum / outer_sum
        else:
            central_conc = 0
        
        return central_conc
    except Exception as e:
        return 0

def calc_spiral_arm_strength(data):
    """
    Measures strength of spiral arm pattern
    Uses multiple scale smoothing to detect arm structures
    """
    try:
        # Smooth at different scales
        smooth_small = gaussian_filter(data, sigma=3)
        smooth_large = gaussian_filter(data, sigma=10)
        
        # Difference highlights medium-scale structures (spiral arms)
        arm_structure = smooth_small - smooth_large
        arm_structure[arm_structure < 0] = 0
        
        total = np.sum(data)
        if total > 0:
            arm_strength = np.sum(arm_structure) / total
        else:
            arm_strength = 0
        
        return arm_strength
    except Exception as e:
        return 0

def calc_gini_coefficient(data):
    """
    Gini coefficient - measures light distribution inequality
    Higher values indicate more concentrated light distribution
    """
    try:
        flat_data = data.flatten()
        flat_data = flat_data[flat_data > 0]  # Remove zeros
        
        if len(flat_data) == 0:
            return 0
        
        sorted_data = np.sort(flat_data)
        n = len(sorted_data)
        index = np.arange(1, n + 1)
        
        gini = (2 * np.sum(index * sorted_data)) / (n * np.sum(sorted_data)) - (n + 1) / n
        
        return gini
    except Exception as e:
        return 0

def calc_m20_moment(data):
    """
    M20 moment - second order moment of brightest 20% of pixels
    Useful for morphology classification
    """
    try:
        cy, cx = data.shape[0] // 2, data.shape[1] // 2
        
        # Get brightest 20% of pixels
        threshold = np.percentile(data, 80)
        bright_mask = data >= threshold
        
        if not np.any(bright_mask):
            return 0
        
        # Calculate second moment
        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        
        total_bright = np.sum(data[bright_mask])
        if total_bright == 0:
            return 0
        
        moment = np.sum(data[bright_mask] * ((x[bright_mask] - cx)**2 + (y[bright_mask] - cy)**2))
        total_moment = np.sum(data * ((x - cx)**2 + (y - cy)**2))
        
        if total_moment > 0:
            m20 = np.log10(moment / total_moment)
        else:
            m20 = 0
        
        return m20
    except Exception as e:
        return 0

def calc_ellipticity(data):
    """
    Overall ellipticity of the galaxy
    Barred spirals may show higher ellipticity
    """
    try:
        cy, cx = data.shape[0] // 2, data.shape[1] // 2
        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        
        total = np.sum(data)
        if total == 0:
            return 0
        
        # Second moments
        mxx = np.sum(data * (x - cx)**2) / total
        myy = np.sum(data * (y - cy)**2) / total
        mxy = np.sum(data * (x - cx) * (y - cy)) / total
        
        # Ellipticity
        trace = mxx + myy
        det = mxx * myy - mxy**2
        
        if det > 0 and trace > 0:
            ellipticity = 1 - np.sqrt(4 * det / trace**2)
        else:
            ellipticity = 0
        
        return ellipticity
    except Exception as e:
        return 0

def calc_petrosian_radius(data):
    """
    Petrosian radius - radius where surface brightness profile drops
    Different for barred vs unbarred spirals
    """
    try:
        cy, cx = data.shape[0] // 2, data.shape[1] // 2
        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        r = np.sqrt((x - cx)**2 + (y - cy)**2)
        
        max_r = min(cx, cy)
        if max_r < 10:
            return 0
        
        radii = np.linspace(0, max_r, 50)
        
        eta_values = []
        for radius in radii[1:]:
            annulus_mask = (r >= radius - max_r/50) & (r < radius)
            circle_mask = r < radius
            
            if np.any(annulus_mask) and np.any(circle_mask):
                annulus_brightness = np.mean(data[annulus_mask])
                circle_brightness = np.mean(data[circle_mask])
                
                if circle_brightness > 0:
                    eta = annulus_brightness / circle_brightness
                    eta_values.append(eta)
                else:
                    eta_values.append(0)
            else:
                eta_values.append(0)
        
        # Petrosian radius where eta = 0.2
        eta_values = np.array(eta_values)
        if len(eta_values) > 0 and np.any(eta_values < 0.2):
            petro_idx = np.where(eta_values < 0.2)[0][0]
            petrosian_r = radii[petro_idx + 1]
        else:
            petrosian_r = max_r
        
        return petrosian_r / max_r  # Normalized
    except Exception as e:
        return 0

# Process all preprocessed files
fits_files = glob.glob(os.path.join(input_folder, "*.fits"))
total = len(fits_files)
print(f"Extracting enhanced features from {total} files")
print("="*60)

features_list = []
error_count = 0

for i, fpath in enumerate(fits_files, 1):
    fname = os.path.basename(fpath)
    
    if i % max(1, total // 10) == 0:
        print(f"Progress: {i}/{total} ({100*i//total}%) - Errors: {error_count}")
    
    try:
        with fits.open(fpath) as hdul:
            # Get primary data
            if len(hdul) > 0:
                data = hdul[0].data
            else:
                raise ValueError("No data in FITS file")
            
            # Validate data
            validate_data(data, fname)
            
            # Normalize data for consistent measurements
            if data.max() > 0:
                data = data.astype(float) / data.max()
            else:
                raise ValueError("Max value is zero")
            
            # Create rings
            rings = create_rings(data, n_rings=10)
            
            # Calculate all features
            da = calc_descent_average(data, rings)
            dv = calc_descent_variance(data, rings)
            asym_180, asym_90 = calc_asymmetry(data)  # Both angles
            clump = calc_clumpiness(data)
            conc, r50, r90 = calc_concentration(data)
            
            # Bar-specific features
            bar_bulge = calc_bar_to_bulge_ratio(data)
            central_conc = calc_central_concentration(data)
            spiral_strength = calc_spiral_arm_strength(data)
            
            # Additional morphological features
            gini = calc_gini_coefficient(data)
            m20 = calc_m20_moment(data)
            ellip = calc_ellipticity(data)
            petro = calc_petrosian_radius(data)
            
            # NEW: Advanced bar detection features
            color_proxy = calc_color_proxy(data)
            gas_conc = calc_gas_concentration(data)
            radial_slope = calc_radial_profile_slope(data)
            bar_fourier = calc_bar_strength_fourier(data)
            spiral_pitch = calc_spiral_arm_pitch_angle(data)
            bulge_total = calc_bulge_to_total(data)
            ang_momentum = calc_angular_momentum_proxy(data)
            
            features_list.append({
                'filename': fname,
                # Original features
                'DA': da,
                'DV': dv,
                'Asymmetry_180': asym_180,
                'Asymmetry_90': asym_90,
                'Clumpiness': clump,
                'Concentration': conc,
                'R50': r50,
                'R90': r90,
                # Bar-specific features
                'Bar_Bulge_Ratio': bar_bulge,
                'Central_Concentration': central_conc,
                'Spiral_Arm_Strength': spiral_strength,
                # Additional morphological features
                'Gini': gini,
                'M20': m20,
                'Ellipticity': ellip,
                'Petrosian_Radius': petro,
                # Advanced bar detection features
                'Color_Proxy': color_proxy,
                'Gas_Concentration': gas_conc,
                'Radial_Slope': radial_slope,
                'Bar_Fourier_Strength': bar_fourier,
                'Spiral_Pitch_Angle': spiral_pitch,
                'Bulge_To_Total': bulge_total,
                'Angular_Momentum_Proxy': ang_momentum
            })
            
    except Exception as e:
        error_count += 1
        if error_count <= 10:  # Print first 10 errors only
            print(f"  ERROR: {fname} - {str(e)[:80]}")
        elif error_count == 11:
            print(f"  ... suppressing further error messages ...")

# Save features to CSV
df = pd.DataFrame(features_list)
csv_path = os.path.join(output_folder, "galaxy_features_enhanced.csv")
df.to_csv(csv_path, index=False)

print("\n" + "="*60)
print("Enhanced feature extraction complete!")
print(f"Successfully extracted: {len(df)} feature vectors")
print(f"Failed: {error_count} files")
print(f"Success rate: {100*len(df)/total:.1f}%")
print(f"\nFeatures extracted:")
print("  - Original: DA, DV, Asymmetry, Clumpiness, Concentration, R50, R90")
print("  - Bar-specific: Bar_Bulge_Ratio, Central_Concentration, Spiral_Arm_Strength")
print("  - Additional: Gini, M20, Ellipticity, Petrosian_Radius")
print(f"\nTotal features: {len(df.columns) - 1}")
print(f"\nSaved to: {csv_path}")
print(f"\nSample statistics:")
print(df.describe())

Extracting enhanced features from 467 files
Progress: 46/467 (9%) - Errors: 0
Progress: 92/467 (19%) - Errors: 0
Progress: 138/467 (29%) - Errors: 0
Progress: 184/467 (39%) - Errors: 0
Progress: 230/467 (49%) - Errors: 0
Progress: 276/467 (59%) - Errors: 0
Progress: 322/467 (68%) - Errors: 0
Progress: 368/467 (78%) - Errors: 0
Progress: 414/467 (88%) - Errors: 0
Progress: 460/467 (98%) - Errors: 0

Enhanced feature extraction complete!
Successfully extracted: 467 feature vectors
Failed: 0 files
Success rate: 100.0%

Features extracted:
  - Original: DA, DV, Asymmetry, Clumpiness, Concentration, R50, R90
  - Bar-specific: Bar_Bulge_Ratio, Central_Concentration, Spiral_Arm_Strength
  - Additional: Gini, M20, Ellipticity, Petrosian_Radius

Total features: 22

Saved to: featurestrain\galaxy_features_enhanced.csv

Sample statistics:
               DA          DV  Asymmetry_180  Asymmetry_90  Clumpiness  \
count  467.000000  467.000000     467.000000    467.000000  467.000000   
mean     0.3